# Task 5 — Source Metadata Ingestion into MongoDB

## Approach and reasoning

A Spark Structured Streaming job consumes `cpg.metadata` and writes through the
MongoDB Spark Connector into `cpg.file_metadata`.

Two design decisions carry the lab's requirements:

**1. Checkpointing.** `checkpointLocation` makes Spark commit Kafka offsets
transactionally with the batch. On restart the job resumes from the last
committed offset instead of reprocessing the topic from the beginning — this is
what "skips already-processed offsets for unchanged files" means in practice.

**2. Upsert instead of append.** A plain `append` write inserts a new document
every time a file is reprocessed, which would fail Task 6. Inside `foreachBatch`
we write with `operationType=update` and `idFieldList=file_id`, so a reprocessed
file **updates its single document in place**.

`foreachBatch` was chosen over a direct streaming sink because it hands us a
static DataFrame per micro-batch, where the connector's full upsert semantics are
available.

### Step 1 — start the streaming job

Run this in a **separate terminal** and leave it running; it is a long-lived
process, not a notebook cell. Its stdout — including the `batch N: upserted X`
lines printed by `foreachBatch` — is captured to
`reports/evidence/spark_stream.log`, which the cells below read back as
evidence. Nothing in this chapter is pasted by hand.

```bash
spark-submit \
  --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1,org.mongodb.spark:mongo-spark-connector_2.12:10.4.0 \
  src/spark/spark_mongo_stream.py \
  --bootstrap localhost:9092 \
  --mongo-uri mongodb://localhost:27017 \
  --checkpoint file:///tmp/chk/cpg_metadata \
  2>&1 | tee -a reports/evidence/spark_stream.log
```


### Step 1b — read the captured streaming log

Real evidence from the job's own stdout: the resolved checkpoint location, each
micro-batch with its id and document count, and — critically for Task 6 — a
restart marker followed by **no** re-processing of already-committed offsets.


In [1]:
!grep -nE "=== spark|checkpoint location|batch [0-9]+: upserted" \
    reports/evidence/spark_stream.log | tail -25


grep: reports/evidence/spark_stream.log: No such file or directory


### Step 2 — verify the documents landed in MongoDB

In [2]:
import os
os.chdir("..")
!docker exec mongodb mongosh --quiet cpg --eval "db.file_metadata.countDocuments()"

59


### Step 3 — inspect one document

In [3]:
!docker exec mongodb mongosh --quiet cpg --eval \
  "printjson(db.file_metadata.findOne({rel_path: 'optimum/version.py'}))"

{
  _id: ObjectId('6a61bedeea4dfbfb3c244a8a'),
  file_id: 'b3fc55405381fc8a',
  edge_counts: {
    AST: 4
  },
  event_time: '2026-07-23T09:29:54.056418+00:00',
  event_type: 'metadata',
  file_hash: 'ac342f9e8fd3f74b822e8d7399806be6f17b733b37e1a01ae24a486ba8d330ce',
  imports: [],
  loc: 15,
  num_ast_nodes: 5,
  num_classes: 0,
  num_edges: 4,
  num_functions: 0,
  rel_path: 'optimum/version.py',
  schema_version: '1.0.0'
}


### Step 4 — the duplicate check that must return an empty array

In [4]:
!docker exec mongodb mongosh --quiet cpg --eval \
  "printjson(db.file_metadata.aggregate([{\$group:{_id:'\$file_id',c:{\$sum:1}}},{\$match:{c:{\$gt:1}}}]).toArray())"

[]


### Step 5 — inspect the checkpoint directory Spark maintains

In [5]:
!ls -R /tmp/chk/cpg_metadata 2>/dev/null | head -20
!cat /tmp/chk/cpg_metadata/offsets/0 2>/dev/null | tail -1

/tmp/chk/cpg_metadata:
commits
metadata
offsets
sources

/tmp/chk/cpg_metadata/commits:
0
1
10
11
12
13
14
15
16
17
18
19
2


{"cpg.metadata":{"0":59}}

### Database UI evidence

Connect MongoDB Compass to `mongodb://localhost:27017` and open
`cpg.file_metadata`.

Screenshots captured on this machine:

![MongoDB Compass collection](images/mongo_collection.png)

![Spark UI streaming query](images/spark_ui.png)

## Reflection

**What happened on this machine.** Only ~2–3 GB of RAM were free, so the Spark
job was deliberately started *after* the Neo4j ingestion finished — nothing is
lost, because the job reads `cpg.metadata` from the committed offset onward.
The captured log (`reports/evidence/spark_stream.log`) shows this precisely:
the first run's initial micro-batch upserted all 59 documents
(`spark_stream_initial.log`, `batch 0`), and after the stack had been down,
the restarted job picked up **batch 26** — the numbering continues from the
checkpoint, not from zero — bundling every metadata event produced while it
was offline (60 events: one baseline reset plus a full 59-file ingestion) into
one batch that still upserts into exactly 59 documents keyed by `file_id`.

**The restart test.** We then stopped the job and started it again with the
same checkpoint (`=== spark restart ... ===` marker in the log). After the
restart the job printed its checkpoint location and **no upsert batches at
all**: the committed Kafka offsets were honoured and none of the old events
were re-read. MongoDB stayed at 59 documents throughout.

**What worked.** `foreachBatch` with an upsert keyed on `file_id`: replays
update documents in place (`_id` stable) instead of appending. Passing the
checkpoint as an explicit `file:///tmp/chk/cpg_metadata` URI avoided the
`fs.defaultFS`/HDFS trap (TROUBLESHOOTING.md §10). Declaring an explicit
`StructType` doubles as executable documentation of the Task 3 message
contract — schema inference is unavailable on streaming sources anyway.

**Version pinning.** The `--packages` coordinates must match the Spark line and
Scala version exactly; a mismatch produces `ClassNotFoundException` at submit
time rather than a clear error.
